In [1]:
import pandas as pd
import numpy as np
import os

# =========================================================================
# 1. LOAD DATASET FROM SPECIFIC PATH
# =========================================================================
input_path = r"C:\Project\data\processed\flower_prices_cleaned.csv"
df = pd.read_csv(input_path)

df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)

price_col = 'Price'  # Adjust to match your actual price column name if needed

# Ensure 'is_festival' flag exists
if 'is_festival' not in df.columns:
    df['is_festival'] = df['Festival Name'].notna().astype(int)

# =========================================================================
# 2. CALCULATE RELATIVE DAY OFFSET (-7 TO +7)
# =========================================================================
df['rel_day'] = np.nan
festival_indices = df[df['is_festival'] == 1].index

for f_idx in festival_indices:
    start_idx = max(0, f_idx - 7)
    end_idx = min(len(df) - 1, f_idx + 7)
    
    for idx in range(start_idx, end_idx + 1):
        offset = idx - f_idx  # Yields values from -7 to +7
        
        current_val = df.loc[idx, 'rel_day']
        if pd.isna(current_val) or abs(offset) < abs(current_val):
            df.loc[idx, 'rel_day'] = offset

# =========================================================================
# 3. CREATE BINARY DUMMY FEATURES FOR ML MODELS
# =========================================================================
for offset in range(-7, 8):
    if offset < 0:
        col_name = f'rel_day_{offset}'
    elif offset > 0:
        col_name = f'rel_day_p{offset}'
    else:
        col_name = 'rel_day_0'
        
    df[col_name] = (df['rel_day'] == offset).astype(int)

df['in_festival_window'] = df['rel_day'].notna().astype(int)

# =========================================================================
# 4. SAVE TO SPECIFIC PROCESSED DIRECTORY
# =========================================================================
output_dir = r"C:\Project\data\processed"
os.makedirs(output_dir, exist_ok=True)  # Ensures folder exists

output_path = os.path.join(output_dir, "flower_prices_with_rel_day.csv")
df.to_csv(output_path, index=False)

print(f"File successfully saved at: {output_path}")

File successfully saved at: C:\Project\data\processed\flower_prices_with_rel_day.csv
